Notebook is used to work with Deployment scripts using PowerShell and Azure CLI when needed.  If you would like to use Cloud Shell in Notebook see ref:  [Title](https://github.com/dotnet/interactive/blob/main/samples/notebooks/powershell/Docs/Interact-With-Azure-Cloud-Shell.ipynb)

In [1]:
#Login to Azure
#Use -Environment Parm to if you need to use Azure Gov.  AzureUSGovernmnet

Connect-AzAccount


To override which subscription Connect-AzAccount selects by default, use `Update-AzConfig -DefaultSubscriptionForLogin 00000000-0000-0000-0000-000000000000`. Go to https://go.microsoft.com/fwlink/?linkid=2200610 for more information.

Account                SubscriptionName TenantId                             Environment
-------                ---------------- --------                             -----------
frgarofa@microsoft.com DSE EDog         72f988bf-86f1-41af-91ab-2d7cd011db47 AzureCloud



In [2]:
#Select Subscription

#$subscription = "Azure FFL Internal Subscription FRGAROFA (GOV)"

$subscription = "Azure FFL Internal Subscription FRGAROFA"

Select-AzSubscription -Subscription $subscription



Name                                     Account        SubscriptionNa Environment    TenantId
                                                        me
----                                     -------        -------------- -----------    --------
Azure FFL Internal Subscription FRGAROF… frgarofa@micr… Azure FFL Int… AzureCloud     72f988bf-86f…



In [3]:
#Create Resource Group if not setup for data maangment zone:

$ResourceGroup = 'dmz-cloudscale-dev'

New-AzResourceGroup -Name $ResourceGroup -Location 'East US'


Confirm
Provided resource group already exists. Are you sure you want to update it?
[Y] Yes  [N] No  [S] Suspend  [?] Help(default is 'Y')

Error: Command cancelled.

In [3]:
#Check Resource Group 

$ResourceGroup = 'dmz-cloudscale-dev'

Get-AzResource -ResourceGroupName $ResourceGroup | Format-Table -AutoSize


Name                                                                               ResourceGroupNam
                                                                                   e
----                                                                               ----------------
purviewacct-5h7tsge77gt7i-dev                                                      dmz-cloudscale-…
purviewacct-5h7tsge77gt7i-account-pep-dev.nic.880b6bb7-4341-4ff2-a6e9-1e3f7b32456e dmz-cloudscale-…
purviewacct-5h7tsge77gt7i-portal-pep-dev.nic.6fc3c8f6-477e-4cc2-98db-e15f85500ef6  dmz-cloudscale-…
purviewacct-5h7tsge77gt7i-account-pep-dev                                          dmz-cloudscale-…
purviewacct-5h7tsge77gt7i-portal-pep-dev                                           dmz-cloudscale-…



In [8]:
#Set Param values for ARM deployment template

$params = @{
    ResourceGroupName = 'dmz-cloudscale-dev'
    TemplateFile = '..\arm\DMZ\Purview\Purview.template.json'
    TemplateParameterFile = '..\arm\DMZ\Purview\Purview.parameters.json'
    Verbose = $true
}

In [9]:
#Test ARM deplkoyment
Test-AzResourceGroupDeployment @params



SubscriptionNotFound - The subscription '02cd7474-626e-4aa4-b21f-4856bdb92779' could not be found.



In [ ]:
#Deploy ARM Template

New-AzResourceGroupDeployment @params

In [ ]:
#Get list of Supported API versions for resource ProviderNamespace

Get-AzResourceProvider -ProviderNamespace Microsoft.Purview | Select-Object ProviderNamespace -ExpandProperty ResourceTypes | Select-Object * -ExpandProperty ApiVersions | Sort-Object -Descending

In [13]:
#Deploy Private DNS Zones

$params = @{
    ResourceGroupName = 'demo-core-vnet'
    TemplateFile = '..\bicep\DMZ\services\Network\privateDnsZones\privateDnsZones.bicep'
}

#Test-AzResourceGroupDeployment @params

New-AzResourceGroupDeployment @params -Verbose



VERBOSE: Using Bicep v0.16.1
VERBOSE: Performing the operation "Creating Deployment" on target "demo-core-vnet".
VERBOSE: 2:53:16 PM - Template is valid.
VERBOSE: 2:53:17 PM - Create template deployment 'privateDnsZones'
VERBOSE: 2:53:17 PM - Checking deployment status in 5 seconds
VERBOSE: 2:53:22 PM - Resource Microsoft.Resources/deploymentScripts 'GetExistingVNetLinks' provisioning status is running
VERBOSE: 2:53:22 PM - Checking deployment status in 14 seconds
VERBOSE: 2:53:36 PM - Checking deployment status in 15 seconds
VERBOSE: 2:53:51 PM - Checking deployment status in 16 seconds
VERBOSE: 2:54:08 PM - Checking deployment status in 15 seconds
VERBOSE: 2:54:23 PM - Checking deployment status in 15 seconds
VERBOSE: 2:54:38 PM - Resource Microsoft.Resources/deploymentScripts 'GetExistingVNetLinks' provisioning status is succeeded
VERBOSE: 2:54:38 PM - Checking deployment status in 5 seconds
VERBOSE: 2:54:48 PM - Resource Microsoft.Resources/deployments 'privatelink.queue.core.windo

In [14]:
#Test and Deploy with Bicep

$params = @{
    ResourceGroupName = 'dmz-cloudscale-dev'
    TemplateFile = '..\bicep\DMZ\services\Purview\Purview.bicep'
    Verbose = $true
}
#Test-AzResourceGroupDeployment @params

New-AzResourceGroupDeployment @params

VERBOSE: Using Bicep v0.16.1
S:\Repos\GitHub\csa-inabox\deploy\bicep\DMZ\services\Purview\Purview.bicep(85,39) : Warning no-hardcoded-env-urls: Environment URLs should not be hardcoded. Use the environment() function to ensure compatibility across clouds. Found this disallowed host: "core.windows.net" [https://aka.ms/bicep/linter/no-hardcoded-env-urls]
S:\Repos\GitHub\csa-inabox\deploy\bicep\DMZ\services\Purview\Purview.bicep(107,22) : Warning BCP081: Resource type "Microsoft.Purview/accounts@2021-12-01" does not have types available.
S:\Repos\GitHub\csa-inabox\deploy\bicep\DMZ\services\Purview\Purview.bicep(122,24) : Warning BCP081: Resource type "Microsoft.Purview/accounts/kafkaConfigurations@2021-12-01" does not have types available.
S:\Repos\GitHub\csa-inabox\deploy\bicep\DMZ\services\Purview\Purview.bicep(140,33) : Warning BCP081: Resource type "Microsoft.Network/privateEndpoints@2023-04-01" does not have types available.
S:\Repos\GitHub\csa-inabox\deploy\bicep\DMZ\services\Purvie

In [66]:
$zones = [string[]]( 'privatelink.purviewstudio.azure.com','privatelink.dfs.core.windows.net')
$rg = 'demo-core-vnet'


function GetExistingVNetLinks {
param([string[]] $zones, [string] $ResourceGroupName)
 foreach ($zone in $zones) {
            $existingLinks += Get-AzPrivateDnsVirtualNetworkLink -ZoneName $zone -ResourceGroupName $ResourceGroupName -ErrorAction 'Ignore' | Select ZoneName, @{label='vNetName'; expression={$_.VirtualNetworkId.Split('/')[-1]}} -erroraction silentlycontinue 
    }
   $existingLinks
}

GetExistingVNetLinks -zones $zones -ResourceGroupName $rg -ErrorAction 'Ignore'




ZoneName                            vNetName
--------                            --------
privatelink.purviewstudio.azure.com demo-core-vnet
privatelink.purviewstudio.azure.com demo-core-vnet-east
privatelink.purviewstudio.azure.com demo-privatelink-vnet
privatelink.dfs.core.windows.net    demo-core-vnet



In [58]:
Get-AzVirtualNetwork -Name 'demo-databricks-vnet'


ResourceGroupName          Name                 Location ProvisioningState EnableDdosProtection
-----------------          ----                 -------- ----------------- --------------------
demo-databricks-secure-dev demo-databricks-vnet eastus   Succeeded         False

